# Análisis de la Capa Gold

## BootCamp de Ingeniería de Datos

Este notebook consume las tablas analíticas almacenadas en PostgreSQL y tiene cuatro objetivos:

1. Validar la disponibilidad de la capa Gold.
2. Calcular KPIs académicos, financieros, de suscripciones y CRM.
3. Generar visualizaciones e insights para Power BI.
4. Documentar limitaciones y decisiones de negocio.

> Los importes monetarios se analizan siempre por moneda. No se suman monedas distintas.

## 1. Arquitectura analítica

```text
CSV Raw
   ↓
Bronze Parquet
   ↓
Silver Parquet
   ↓
Gold PostgreSQL
   ↓
Jupyter / Power BI / SQL
```

Tablas disponibles:

- `gold.academic_performance`
- `gold.invoice_financial`
- `gold.product_sales`
- `gold.subscription_portfolio`
- `gold.crm_opportunity`
- `gold.crm_lead`
- `gold.student_360`

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from sqlalchemy import inspect, text


def find_project_root(start: Path | None = None) -> Path:
    """Locate the repository root from local or Docker Jupyter."""

    current = (start or Path.cwd()).resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "src").exists() and (candidate / "docker-compose.yml").exists():
            return candidate

    raise FileNotFoundError(
        "No se encontró la raíz del proyecto. "
        "Ejecuta el notebook dentro del repositorio."
    )


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))

from src.database.connection import get_postgres_engine

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

print(f"Raíz del proyecto: {PROJECT_ROOT}")

FileNotFoundError: No se encontró la raíz del proyecto. Ejecuta el notebook dentro del repositorio.

## 2. Conexión con PostgreSQL

La conexión reutiliza las variables de entorno configuradas para el proyecto.

- Desde Windows: `POSTGRES_HOST=localhost`
- Desde Docker: `POSTGRES_HOST=postgres`

In [ ]:
engine = get_postgres_engine()

connection_query = text(
    """
    SELECT
        current_database() AS database_name,
        current_user AS database_user,
        current_schema() AS active_schema
    """
)

with engine.connect() as connection:
    connection_information = connection.execute(
        connection_query
    ).mappings().one()

display(pd.DataFrame([dict(connection_information)]))

## 3. Inventario de tablas Gold

Esta comprobación valida que las siete tablas esperadas se encuentren disponibles en PostgreSQL.

In [ ]:
EXPECTED_GOLD_TABLES = [
    "academic_performance",
    "invoice_financial",
    "product_sales",
    "subscription_portfolio",
    "crm_opportunity",
    "crm_lead",
    "student_360",
]

inspector = inspect(engine)
available_gold_tables = sorted(
    inspector.get_table_names(schema="gold")
)

inventory_rows = []

for table_name in EXPECTED_GOLD_TABLES:
    row_count_query = text(
        f'SELECT COUNT(*) AS row_count FROM gold."{table_name}"'
    )

    with engine.connect() as connection:
        row_count = int(
            connection.execute(row_count_query).scalar_one()
        )

    inventory_rows.append(
        {
            "table_name": table_name,
            "is_available": table_name in available_gold_tables,
            "row_count": row_count,
        }
    )

gold_inventory = pd.DataFrame(inventory_rows)
display(gold_inventory)

assert gold_inventory["is_available"].all(), (
    "Falta una o más tablas Gold."
)

## 4. Carga de las tablas

Para el volumen actual es posible cargar las siete tablas en memoria. En un entorno de mayor escala, los cálculos deberían ejecutarse directamente en SQL o mediante procesamiento distribuido.

In [ ]:
def read_gold_table(table_name: str) -> pd.DataFrame:
    """Read one Gold table from PostgreSQL."""

    query = text(f'SELECT * FROM gold."{table_name}"')

    return pd.read_sql_query(query, engine)


academic = read_gold_table("academic_performance")
invoices = read_gold_table("invoice_financial")
sales = read_gold_table("product_sales")
subscriptions = read_gold_table("subscription_portfolio")
opportunities = read_gold_table("crm_opportunity")
leads = read_gold_table("crm_lead")
students = read_gold_table("student_360")

print("Tablas Gold cargadas correctamente.")

# 5. KPIs académicos

La tabla `academic_performance` tiene una fila por inscripción.

Los principales indicadores son:

- Tasa de finalización.
- Tasa de reprobación.
- Tasa de abandono.
- Nota normalizada promedio.
- Inscripciones con pesos inconsistentes.

In [ ]:
academic_status = (
    academic["enrollment_status"]
    .value_counts()
    .rename_axis("enrollment_status")
    .reset_index(name="enrollments")
)

total_enrollments = len(academic)
status_counts = academic["enrollment_status"].value_counts()

academic_kpis = pd.DataFrame(
    [
        {
            "total_enrollments": total_enrollments,
            "distinct_students": academic["student_id"].nunique(),
            "completion_rate": (
                status_counts.get("completed", 0)
                / total_enrollments
            ),
            "failure_rate": (
                status_counts.get("failed", 0)
                / total_enrollments
            ),
            "dropout_rate": (
                status_counts.get("dropped", 0)
                / total_enrollments
            ),
            "active_rate": (
                status_counts.get("active", 0)
                / total_enrollments
            ),
            "average_normalized_grade": (
                academic.loc[
                    academic["has_grades"],
                    "normalized_grade",
                ].mean()
            ),
            "invalid_weight_enrollments": int(
                academic["has_invalid_weight_sum"].sum()
            ),
        }
    ]
)

display(academic_kpis)
display(academic_status)

In [ ]:
plot_data = academic_status.sort_values(
    "enrollments",
    ascending=True,
)

plt.figure(figsize=(9, 5))
plt.barh(
    plot_data["enrollment_status"],
    plot_data["enrollments"],
)
plt.title("Inscripciones por estado")
plt.xlabel("Cantidad de inscripciones")
plt.ylabel("Estado")
plt.tight_layout()
plt.show()

### Rendimiento por departamento

La nota se calcula únicamente para inscripciones que tienen calificaciones.

La fórmula normalizada utilizada en Gold es:

```text
SUM(score × weight) / SUM(weight)
```

In [ ]:
department_performance = (
    academic.loc[academic["has_grades"]]
    .groupby("department", as_index=False)
    .agg(
        enrollments=("enrollment_id", "count"),
        students=("student_id", "nunique"),
        average_grade=("normalized_grade", "mean"),
        completion_rate=(
            "enrollment_status",
            lambda values: (values == "completed").mean(),
        ),
    )
    .sort_values("average_grade", ascending=False)
)

display(department_performance)

In [ ]:
plot_data = department_performance.sort_values(
    "average_grade",
    ascending=True,
)

plt.figure(figsize=(10, 6))
plt.barh(
    plot_data["department"],
    plot_data["average_grade"],
)
plt.title("Nota normalizada promedio por departamento")
plt.xlabel("Nota normalizada promedio")
plt.ylabel("Departamento")
plt.tight_layout()
plt.show()

# 6. KPIs de facturación

Los importes se muestran por moneda. No se calcula un total global mezclando USD, CLP, EUR, MXN, COP, ARS, PEN y BRL.

In [ ]:
invoice_status_counts = (
    invoices["invoice_status"]
    .value_counts()
    .rename_axis("invoice_status")
    .reset_index(name="invoice_count")
)

billing_by_currency = (
    invoices.groupby("currency", as_index=False)
    .agg(
        invoice_count=("invoice_id", "count"),
        invoiced_amount=("invoice_total", "sum"),
        paid_amount=("paid_amount", "sum"),
        outstanding_amount=("outstanding_amount", "sum"),
        overpayment_amount=("overpayment_amount", "sum"),
        overdue_invoices=(
            "invoice_status",
            lambda values: (values == "overdue").sum(),
        ),
        invoices_without_items=(
            "has_invoice_items",
            lambda values: (~values.astype(bool)).sum(),
        ),
        invoices_without_payments=(
            "has_payments",
            lambda values: (~values.astype(bool)).sum(),
        ),
        item_mismatches=(
            "invoice_items_match_header",
            lambda values: (~values.astype(bool)).sum(),
        ),
    )
)

billing_by_currency["overdue_rate"] = (
    billing_by_currency["overdue_invoices"]
    / billing_by_currency["invoice_count"]
)

display(invoice_status_counts)
display(billing_by_currency)

In [ ]:
plot_data = invoice_status_counts.sort_values(
    "invoice_count",
    ascending=True,
)

plt.figure(figsize=(8, 4))
plt.barh(
    plot_data["invoice_status"],
    plot_data["invoice_count"],
)
plt.title("Facturas por estado")
plt.xlabel("Cantidad de facturas")
plt.ylabel("Estado")
plt.tight_layout()
plt.show()

In [ ]:
plot_data = billing_by_currency.sort_values(
    "invoice_count",
    ascending=True,
)

plt.figure(figsize=(9, 5))
plt.barh(
    plot_data["currency"],
    plot_data["invoice_count"],
)
plt.title("Cantidad de facturas por moneda")
plt.xlabel("Cantidad de facturas")
plt.ylabel("Moneda")
plt.tight_layout()
plt.show()

## 7. Ventas por producto

`product_sales` tiene una fila por línea de factura. Los rankings monetarios deben filtrarse por una moneda específica.

In [ ]:
selected_currency = (
    sales["currency"]
    .value_counts()
    .index[0]
)

top_products = (
    sales.loc[sales["currency"] == selected_currency]
    .groupby(
        [
            "product_id",
            "sku",
            "product_name",
            "product_category",
        ],
        as_index=False,
    )
    .agg(
        units_sold=("quantity", "sum"),
        sales_amount=("line_total", "sum"),
        invoice_count=("invoice_id", "nunique"),
    )
    .sort_values("sales_amount", ascending=False)
    .head(10)
)

print(
    "Moneda seleccionada automáticamente para el ejemplo:",
    selected_currency,
)
display(top_products)

In [ ]:
plot_data = top_products.sort_values(
    "sales_amount",
    ascending=True,
)

plt.figure(figsize=(11, 7))
plt.barh(
    plot_data["product_name"],
    plot_data["sales_amount"],
)
plt.title(
    f"Top 10 productos por ventas registradas ({selected_currency})"
)
plt.xlabel(f"Ventas registradas ({selected_currency})")
plt.ylabel("Producto")
plt.tight_layout()
plt.show()

# 8. KPIs de suscripciones

Los indicadores principales son:

- Suscripciones por estado.
- Tasa de suscripciones activas.
- Clientes estudiantes con suscripción activa.
- Fechas finales inválidas.
- Distribución por categoría.

In [ ]:
subscription_status = (
    subscriptions["subscription_status"]
    .value_counts()
    .rename_axis("subscription_status")
    .reset_index(name="subscription_count")
)

subscription_kpis = pd.DataFrame(
    [
        {
            "total_subscriptions": len(subscriptions),
            "active_subscriptions": int(
                subscriptions["is_active"].sum()
            ),
            "active_subscription_rate": (
                subscriptions["is_active"].mean()
            ),
            "student_subscriptions": int(
                subscriptions["is_student_customer"].sum()
            ),
            "invalid_end_dates": int(
                subscriptions["invalid_end_date_flag"].sum()
            ),
            "average_valid_duration_days": (
                subscriptions["duration_days"].mean()
            ),
        }
    ]
)

subscriptions_by_category = (
    subscriptions.groupby(
        "product_category",
        as_index=False,
    )
    .agg(
        subscription_count=("subscription_id", "count"),
        active_subscriptions=(
            "is_active",
            "sum",
        ),
    )
)

subscriptions_by_category["active_rate"] = (
    subscriptions_by_category["active_subscriptions"]
    / subscriptions_by_category["subscription_count"]
)

display(subscription_kpis)
display(subscription_status)
display(subscriptions_by_category)

In [ ]:
plot_data = subscription_status.sort_values(
    "subscription_count",
    ascending=True,
)

plt.figure(figsize=(8, 4))
plt.barh(
    plot_data["subscription_status"],
    plot_data["subscription_count"],
)
plt.title("Suscripciones por estado")
plt.xlabel("Cantidad de suscripciones")
plt.ylabel("Estado")
plt.tight_layout()
plt.show()

# 9. KPIs de CRM

CRM se analiza como un dominio independiente porque no existe una clave confiable para relacionarlo directamente con Universidad o Billing.

In [ ]:
opportunity_stage = (
    opportunities["stage"]
    .value_counts()
    .rename_axis("stage")
    .reset_index(name="opportunity_count")
)

closed_opportunities = opportunities.loc[
    opportunities["is_closed"]
]

opportunity_kpis = pd.DataFrame(
    [
        {
            "total_opportunities": len(opportunities),
            "open_opportunities": int(
                opportunities["is_open"].sum()
            ),
            "closed_opportunities": int(
                opportunities["is_closed"].sum()
            ),
            "won_opportunities": int(
                opportunities["is_won"].sum()
            ),
            "lost_opportunities": int(
                opportunities["is_lost"].sum()
            ),
            "closed_win_rate": (
                closed_opportunities["is_won"].mean()
                if len(closed_opportunities)
                else 0
            ),
            "average_activity_count": (
                opportunities["activity_count"].mean()
            ),
            "opportunities_without_activities": int(
                (~opportunities["has_activities"]).sum()
            ),
            "opportunities_without_contacts": int(
                (~opportunities["has_contacts"]).sum()
            ),
            "invalid_close_dates": int(
                opportunities["invalid_close_date_flag"].sum()
            ),
        }
    ]
)

display(opportunity_kpis)
display(opportunity_stage)

In [ ]:
plot_data = opportunity_stage.sort_values(
    "opportunity_count",
    ascending=True,
)

plt.figure(figsize=(9, 5))
plt.barh(
    plot_data["stage"],
    plot_data["opportunity_count"],
)
plt.title("Oportunidades por etapa")
plt.xlabel("Cantidad de oportunidades")
plt.ylabel("Etapa")
plt.tight_layout()
plt.show()

### Funnel de leads

In [ ]:
lead_status = (
    leads["lead_status"]
    .value_counts()
    .reindex(
        [
            "new",
            "contacted",
            "qualified",
            "converted",
            "lost",
        ],
        fill_value=0,
    )
    .rename_axis("lead_status")
    .reset_index(name="lead_count")
)

lead_kpis = pd.DataFrame(
    [
        {
            "total_leads": len(leads),
            "converted_leads": int(
                leads["is_converted"].sum()
            ),
            "qualified_leads": int(
                leads["is_qualified"].sum()
            ),
            "lost_leads": int(
                leads["is_lost"].sum()
            ),
            "lead_conversion_rate": (
                leads["is_converted"].mean()
            ),
            "average_lead_score": (
                leads["lead_score"].mean()
            ),
            "lead_source_count": (
                leads["lead_source"].nunique()
            ),
        }
    ]
)

conversion_by_source = (
    leads.groupby("lead_source", as_index=False)
    .agg(
        leads=("lead_id", "count"),
        converted=("is_converted", "sum"),
        average_score=("lead_score", "mean"),
    )
)

conversion_by_source["conversion_rate"] = (
    conversion_by_source["converted"]
    / conversion_by_source["leads"]
)

display(lead_kpis)
display(lead_status)
display(
    conversion_by_source.sort_values(
        "conversion_rate",
        ascending=False,
    )
)

In [ ]:
plot_data = lead_status.iloc[::-1]

plt.figure(figsize=(9, 5))
plt.barh(
    plot_data["lead_status"],
    plot_data["lead_count"],
)
plt.title("Leads por estado")
plt.xlabel("Cantidad de leads")
plt.ylabel("Estado")
plt.tight_layout()
plt.show()

# 10. Visión 360 del estudiante

`student_360` combina Universidad y Billing mediante:

```text
university.students.student_id
=
billing.customers.external_ref
```

Los análisis siguientes describen asociaciones observadas. No demuestran causalidad.

In [ ]:
student_360_kpis = pd.DataFrame(
    [
        {
            "total_students": len(students),
            "students_with_academic_activity": int(
                students["has_academic_activity"].sum()
            ),
            "students_with_billing_activity": int(
                students["has_billing_activity"].sum()
            ),
            "students_with_subscription_history": int(
                students["has_subscription_history"].sum()
            ),
            "students_with_active_subscription": int(
                students["has_active_subscription"].sum()
            ),
            "students_with_outstanding_invoices": int(
                students["has_outstanding_invoices"].sum()
            ),
            "students_with_overdue_invoices": int(
                students["has_overdue_invoices"].sum()
            ),
        }
    ]
)

grade_by_subscription = (
    students.loc[
        students["average_normalized_grade"].notna()
    ]
    .groupby(
        "has_active_subscription",
        as_index=False,
    )
    .agg(
        students=("student_id", "count"),
        average_grade=(
            "average_normalized_grade",
            "mean",
        ),
        average_completed_courses=(
            "courses_completed",
            "mean",
        ),
        overdue_student_rate=(
            "has_overdue_invoices",
            "mean",
        ),
    )
)

grade_by_subscription["subscription_group"] = (
    grade_by_subscription["has_active_subscription"]
    .map(
        {
            True: "Con suscripción activa",
            False: "Sin suscripción activa",
        }
    )
)

display(student_360_kpis)
display(grade_by_subscription)

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(
    grade_by_subscription["subscription_group"],
    grade_by_subscription["average_grade"],
)
plt.title("Nota promedio por estado de suscripción")
plt.xlabel("Grupo")
plt.ylabel("Nota normalizada promedio")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 11. Indicadores de calidad visibles en Gold

Gold no oculta las anomalías detectadas. Las convierte en campos consultables para que analistas y responsables de negocio puedan medir su impacto.

In [ ]:
quality_indicators = pd.DataFrame(
    [
        {
            "quality_indicator": "Inscripciones con pesos inconsistentes",
            "affected_rows": int(
                academic["has_invalid_weight_sum"].sum()
            ),
            "source_table": "academic_performance",
        },
        {
            "quality_indicator": "Facturas sin líneas",
            "affected_rows": int(
                (~invoices["has_invoice_items"]).sum()
            ),
            "source_table": "invoice_financial",
        },
        {
            "quality_indicator": "Facturas sin pagos",
            "affected_rows": int(
                (~invoices["has_payments"]).sum()
            ),
            "source_table": "invoice_financial",
        },
        {
            "quality_indicator": "Facturas con diferencia cabecera-líneas",
            "affected_rows": int(
                (~invoices["invoice_items_match_header"]).sum()
            ),
            "source_table": "invoice_financial",
        },
        {
            "quality_indicator": "Suscripciones con fecha final inválida",
            "affected_rows": int(
                subscriptions["invalid_end_date_flag"].sum()
            ),
            "source_table": "subscription_portfolio",
        },
        {
            "quality_indicator": "Oportunidades con fecha de cierre inválida",
            "affected_rows": int(
                opportunities["invalid_close_date_flag"].sum()
            ),
            "source_table": "crm_opportunity",
        },
    ]
)

display(
    quality_indicators.sort_values(
        "affected_rows",
        ascending=False,
    )
)

In [ ]:
plot_data = quality_indicators.sort_values(
    "affected_rows",
    ascending=True,
)

plt.figure(figsize=(11, 6))
plt.barh(
    plot_data["quality_indicator"],
    plot_data["affected_rows"],
)
plt.title("Principales hallazgos de calidad visibles en Gold")
plt.xlabel("Registros afectados")
plt.ylabel("Hallazgo")
plt.tight_layout()
plt.show()

# 12. Insights iniciales

Las siguientes celdas generan hallazgos dinámicos a partir de Gold. Deben revisarse antes de incorporarlos al dashboard o a la presentación.

In [ ]:
best_department = department_performance.iloc[0]

best_lead_source = (
    conversion_by_source
    .sort_values(
        ["conversion_rate", "leads"],
        ascending=[False, False],
    )
    .iloc[0]
)

subscription_grade_lookup = (
    grade_by_subscription
    .set_index("has_active_subscription")
)

active_grade = subscription_grade_lookup.loc[
    True,
    "average_grade",
] if True in subscription_grade_lookup.index else float("nan")

inactive_grade = subscription_grade_lookup.loc[
    False,
    "average_grade",
] if False in subscription_grade_lookup.index else float("nan")

insights = pd.DataFrame(
    [
        {
            "area": "Académico",
            "insight": (
                f"El departamento con mayor nota normalizada promedio es "
                f"{best_department['department']} con "
                f"{best_department['average_grade']:.2f}."
            ),
        },
        {
            "area": "Calidad académica",
            "insight": (
                f"{int(academic['has_invalid_weight_sum'].sum()):,} "
                "inscripciones presentan pesos que no suman 1."
            ),
        },
        {
            "area": "Facturación",
            "insight": (
                f"{int((~invoices['invoice_items_match_header']).sum()):,} "
                "facturas presentan diferencias entre cabecera y líneas."
            ),
        },
        {
            "area": "Suscripciones",
            "insight": (
                f"{subscriptions['is_active'].mean():.2%} de las "
                "suscripciones se encuentran activas."
            ),
        },
        {
            "area": "CRM",
            "insight": (
                f"La fuente con mayor conversión observada es "
                f"{best_lead_source['lead_source']} con "
                f"{best_lead_source['conversion_rate']:.2%}."
            ),
        },
        {
            "area": "Student 360",
            "insight": (
                "La diferencia observada de nota promedio entre estudiantes "
                f"con y sin suscripción activa es "
                f"{active_grade - inactive_grade:.2f} puntos. "
                "Este resultado describe asociación, no causalidad."
            ),
        },
    ]
)

display(insights)

# 13. Propuesta de páginas para Power BI

## Página 1 — Resumen ejecutivo

- Estudiantes.
- Inscripciones.
- Tasa de finalización.
- Facturas vencidas.
- Suscripciones activas.
- Conversión de leads.
- Principales alertas de calidad.

## Página 2 — Rendimiento académico

- Estado de inscripciones.
- Nota por departamento.
- Nota por curso y semestre.
- Inscripciones con pesos inconsistentes.

## Página 3 — Facturación y productos

- Facturación por moneda.
- Estado de facturas.
- Saldos pendientes por moneda.
- Ventas por producto y categoría.
- Alertas de reconciliación.

## Página 4 — Suscripciones y Student 360

- Suscripciones por estado.
- Estudiantes con suscripción activa.
- Rendimiento académico por estado de suscripción.
- Estudiantes con facturas vencidas.
- Segmentos de cliente.

## Página 5 — CRM

- Funnel de leads.
- Conversión por fuente.
- Oportunidades por etapa.
- Win rate.
- Actividades por oportunidad.

# 14. Limitaciones

- Billing contiene ocho monedas y no se proporcionaron tipos de cambio históricos.
- `crm.opportunities.amount` no contiene moneda.
- CRM no tiene una clave confiable para integrarse con Billing o Universidad.
- Los pesos académicos presentan inconsistencias generalizadas.
- Facturas, líneas y pagos contienen diferencias que deben interpretarse como problemas de calidad.
- Los análisis integrados describen asociaciones y no demuestran causalidad.

# 15. Conclusiones

La capa Gold proporciona contratos de consumo claros para analistas y Power BI.

El pipeline:

- Preserva la trazabilidad.
- Expone problemas de calidad.
- Evita mezclar monedas.
- Mantiene CRM separado cuando no existe una clave confiable.
- Permite análisis integrados de estudiantes, facturación y suscripciones.
- Puede reconstruirse mediante Airflow sin producir duplicados.

In [ ]:
validation_summary = pd.DataFrame(
    [
        {
            "check": "Siete tablas Gold disponibles",
            "is_valid": (
                len(gold_inventory) == 7
                and gold_inventory["is_available"].all()
            ),
        },
        {
            "check": "25.000 inscripciones académicas",
            "is_valid": len(academic) == 25_000,
        },
        {
            "check": "50.000 facturas",
            "is_valid": len(invoices) == 50_000,
        },
        {
            "check": "150.000 líneas de producto",
            "is_valid": len(sales) == 150_000,
        },
        {
            "check": "15.000 suscripciones",
            "is_valid": len(subscriptions) == 15_000,
        },
        {
            "check": "3.000 oportunidades",
            "is_valid": len(opportunities) == 3_000,
        },
        {
            "check": "2.000 leads",
            "is_valid": len(leads) == 2_000,
        },
        {
            "check": "5.000 estudiantes",
            "is_valid": len(students) == 5_000,
        },
    ]
)

display(validation_summary)

assert validation_summary["is_valid"].all(), (
    "Una o más validaciones finales no pasaron."
)

print("Notebook Gold ejecutado correctamente.")